# F1 AI Strategy Simulator

## Step 1 — Building the Telemetry Dataset

This notebook collects historical Formula 1 race telemetry using the FastF1 library.

The dataset will be used later for:

- Lap time prediction
- Tire degradation modeling
- Overtake probability modeling
- Race simulation
- Strategy optimization

Each row in the dataset represents **one driver lap**.

The final dataset will be saved to:

data/race_dataset.csv

In [ ]:
import os
import fastf1 
import time
import gc
import pandas as pd
import numpy as np

# Create cache directory
os.makedirs("../data/cache", exist_ok=True)

# Enable FastF1 cache
fastf1.Cache.enable_cache("../data/cache")

## Collect Historical Race Data

We download race telemetry from multiple Formula 1 seasons.

Using multiple seasons improves machine learning model performance
because the models learn from many race situations.

Seasons included in this dataset:

2022  
2023  
2024

In [ ]:
SEASONS = [2025, 2024, 2023, 2022, 2021, 2020, 2019]
# Output dataset file
output_file = "../data/race_dataset.csv"

# Remove existing dataset if present
if os.path.exists(output_file):
    os.remove(output_file)

## Load Race Sessions

For each season we:

1. Retrieve the official race calendar
2. Load each race session
3. Extract lap-level telemetry data

In [ ]:
for season in SEASONS:

    print("\nSeason:", season)

    schedule = fastf1.get_event_schedule(season)

    for _, event in schedule.iterrows():

        try:

            print("Loading:", season, event["EventName"])

            session = fastf1.get_session(
                season,
                event["RoundNumber"],
                "R"
            )

            session.load()
            weather = session.weather_data


            time.sleep(2)

            laps = session.laps

            df = laps[
                [
                    "Driver",
                    "Team",
                    "LapNumber",
                    "LapTime",
                    "Compound",
                    "TyreLife",
                    "Stint",
                    "Position",
                    "TrackStatus",
                    "PitInTime",
                    "PitOutTime",
                    "Sector1Time",
                    "Sector2Time",
                    "Sector3Time"
                ]
            ].copy()
            df["AirTemp"] = weather["AirTemp"].mean()
            df["TrackTemp"] = weather["TrackTemp"].mean()
            df["Humidity"] = weather["Humidity"].mean()
            df["WindSpeed"] = weather["WindSpeed"].mean()

            # Convert time features
            df["LapTimeSeconds"] = df["LapTime"].dt.total_seconds()
            df["Sector1Seconds"] = df["Sector1Time"].dt.total_seconds()
            df["Sector2Seconds"] = df["Sector2Time"].dt.total_seconds()
            df["Sector3Seconds"] = df["Sector3Time"].dt.total_seconds()

            # Remove safety car laps
            df = df[df["TrackStatus"].astype(str).str.contains("1")]

            # Remove missing lap times
            df = df.dropna(subset=["LapTimeSeconds"])

            # Metadata
            df["Season"] = season
            df["Track"] = event["Location"]
            df["RegulationEra"] = np.where(season >= 2022, 1, 0)
            df = df[(df["LapTimeSeconds"] > 40) & (df["LapTimeSeconds"] < 200)]
            # Pit stop indicator
            df["IsPitLap"] = np.where(df["PitInTime"].notna(), 1, 0)

            # Save immediately to avoid RAM overflow
            df.to_csv(
                output_file,
                mode="a",
                header=not os.path.exists(output_file),
                index=False
            )

            print("Saved:", season, event["EventName"], "| Laps:", len(df))

            # Free memory
            del df
            del laps
            del session
            gc.collect()

        except Exception as e:

            print("Skipped:", event["EventName"], e)

## Combine All Races

After collecting lap data from every race we combine
all races into a single dataset.

In [ ]:
df = pd.read_csv("../data/race_dataset.csv")

print("Total laps collected:", len(df))
df.groupby("Season").size()

In [1]:
print(df.shape)

print("Unique drivers:", df["Driver"].nunique())
print("Unique tracks:", df["Track"].nunique())
print("Unique teams:", df["Team"].nunique())

df.head()

NameError: name 'df' is not defined